In [1]:
import pandas as pd
import geopandas as gpd
import os
import numpy as np

epa_cbsa = pd.read_csv("/work/hawkins_lab/vulcan/data/transport/epa_cbsa.csv")

In [2]:
epa_cbsa

,cbsa_cbsa,ac_total_cbsa,ac_water_cbsa,ac_land_cbsa,ac_unpr_cbsa,totpop_cbsa,counthu_cbsa,hh_cbsa,autoown0_cbsa,autoown1_cbsa,...,d3aao_cbsa,d3amm_cbsa,d3apo_cbsa,d3b_cbsa,d3bao_cbsa,d3bmm3_cbsa,d3bmm4_cbsa,d3bpo3_cbsa,d3bpo4_cbsa,d4d_cbsa
0,19100.00,4.000831e+05,1492.012780,3.985911e+05,3.950876e+05,10352,4526.0,3609.0,423,1172,...,0.221626,0.744148,0.246234,0.861672,0.275285,0.794435,0.046870,0.330416,0.064528,-99999.000000
1,33260.00,4.978333e+05,667.466026,4.971658e+05,4.971658e+05,20025,10026.0,6708.0,520,2356,...,0.247069,1.118015,0.256659,1.467715,0.324399,1.248678,0.088823,0.629488,0.126155,-99999.000000
2,26420.00,3.839765e+05,29728.191308,3.542483e+05,3.412167e+05,25853,16531.0,10606.0,435,2747,...,0.181594,1.720599,0.439509,2.948937,0.476336,3.027359,0.218714,0.852236,0.142533,-99999.000000
3,18580.00,5.893370e+05,4730.598290,5.846064e+05,5.836926e+05,13075,7344.0,5350.0,542,1585,...,0.096627,1.004112,0.167333,0.974728,0.174134,1.005793,0.099525,0.227760,0.052422,-99999.000000
4,52.21,8.016300e+05,9058.164866,7.925718e+05,7.656592e+05,24387,12741.0,9358.0,957,2798,...,0.127159,0.830699,0.274302,1.207772,0.235216,1.007028,0.059856,0.541816,0.114837,-99999.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2040,16220.00,1.116337e+06,27016.452156,1.089320e+06,1.044014e+06,545488,259354.0,230243.0,19021,82093,...,0.733400,1.549198,1.736630,8.502273,0.816516,3.292760,0.869477,5.431559,1.813675,-99999.000000
2041,56.27,8.015328e+05,11440.380601,7.900925e+05,7.100745e+05,171365,62685.0,58539.0,3786,16811,...,0.443730,0.136473,2.172301,4.353634,0.661926,0.469395,0.152555,4.300334,1.019670,-64583.672367
2042,56.15,3.532054e+06,3273.739438,3.528780e+06,2.018946e+06,207829,91685.0,72463.0,4328,26404,...,0.074979,0.359675,0.421395,1.893145,0.060688,0.507860,0.196659,1.431641,0.402839,-91174.307575
2043,56.09,4.304457e+05,5145.591620,4.253001e+05,3.531810e+05,86076,37946.0,34239.0,2527,10689,...,0.456017,1.871622,0.610266,4.481564,0.696586,2.897904,0.412167,2.179981,0.682448,-99999.000000


In [ ]:
base_path = '/work/hawkins_lab/'
file_names = ['vulcan_TOT_epa','vulcan_ONR_epa',
               'vulcan_ELC_epa','vulcan_COM_epa',
               'vulcan_IND_epa','vulcan_RES_epa']

for f in file_names:
    print("Starting", f)
    shp_file = os.path.join(base_path,'Mehrnoosh/output/',f+'.shp')
    gdf = gpd.read_file(shp_file, engine="pyogrio", use_arrow=True)
    print("Loaded", shp_file)
    
    # Convert all column names to lowercase
    gdf.columns = gdf.columns.str.lower()
    # Filter out anything where FIPS doesn't match the GEOID
    gdf['fips'] = gdf.geoid10.str[:5].astype(int)

    # Filter out anything outside the continental US
    gdf = gdf[((gdf.fips/1000).astype(int) <=56) & ((gdf.fips/1000).astype(int) !=2) & ((gdf.fips/1000).astype(int) !=15)]
    gdf.loc[gdf.cbsa.isna(),"cbsa"] = (gdf.loc[gdf.cbsa.isna(),"statefp"]).astype(int) + (gdf.loc[gdf.cbsa.isna(),"countyfp"]).astype(int)/100
    gdf.cbsa = round(gdf["cbsa"].astype(float),2)

    gdf = pd.merge(gdf, epa_cbsa, left_on='cbsa', right_on='cbsa_cbsa', how='left')
    par_file = os.path.join(base_path,'vulcan/data/vulcan/parquet',f+'.parquet')
    gdf.to_parquet(par_file)
    print("Done", par_file)
    # print("CO2", gdf.dn_weighte.sum()/10**6)


Starting vulcan_TOT_epa
Loaded /work/hawkins_lab/Mehrnoosh/output/vulcan_TOT_epa.shp
Done /work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_TOT_epa.parquet
Starting vulcan_ONR_epa
Loaded /work/hawkins_lab/Mehrnoosh/output/vulcan_ONR_epa.shp
Done /work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_ONR_epa.parquet
Starting vulcan_ELC_epa
Loaded /work/hawkins_lab/Mehrnoosh/output/vulcan_ELC_epa.shp
Done /work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_ELC_epa.parquet
Starting vulcan_COM_epa
Loaded /work/hawkins_lab/Mehrnoosh/output/vulcan_COM_epa.shp
Done /work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_COM_epa.parquet
Starting vulcan_IND_epa
Loaded /work/hawkins_lab/Mehrnoosh/output/vulcan_IND_epa.shp
Done /work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_IND_epa.parquet
Starting vulcan_RES_epa


In [ ]:
gdf.cbsa.unique().shape

In [ ]:
gdf.cbsa.unique()